# Vajalike pakettide impordid ja seadistused

In [1]:
from typing import Callable

import gspread
import pandas as pd
import duckdb
import re

# Andmete laadimine

In [314]:
# Autentimine
gc = gspread.service_account(filename="freanalyys-732d478588e9.json", scopes=gspread.auth.READONLY_SCOPES)

# Leia ja ava Google Sheets dokument ning selles vajalik tööleht
sh = gc.open("Uuring Tarbijate hoiakud tekstiilide liigiti kogumisel rõivastest ja kodutekstiilidest loobumisel (vastused)")

# Leia vastuste tööleht
vastused = sh.worksheet("Vormi vastused")
# Päri kõik vastused
data_raw  = pd.DataFrame(vastused.get_all_records(default_blank="NULL"))

# Leia vastuste koodide tööleht
v_koodid = sh.worksheet("Vastuste koodid")
# Päri kõik vastuste koodid
vastuste_koodid  = pd.DataFrame(v_koodid.get_all_records())

# Vastuste kodeerimine

In [315]:
data_coded = data_raw.copy()

## Abifunktsioonid

In [265]:
def kodeeri_vastus(kysimus: str):
    '''
    Kodeeri vastused analüüsiks - asenda vastuse tekst vastava koodiga.
    Kui vastus on "muu", salvesta originaaltekst eraldi veergu.
    '''
    # Leia küsimusele vastavad kood-vastus paarid
    mapping = vastuste_koodid[vastuste_koodid['Kysimus'] == kysimus][["Vastus", "Kood"]]
    # Loo dictionary leitud vastus-kood paaridest
    vastavused = mapping.set_index('Vastus')['Kood'].to_dict()
    
    # Leia vastava küsimuse valiku "muu" kood 
    muu_kood = None
    for v, k in vastavused.items():
        if v.strip().lower() == 'muu':
            muu_kood = k
            break
    
    # Kontrolli, kas vastus on vastuste ja koodide tabelist leitav
    def map_answer(vastus):
        # Kontrolli, kas vastus on tühi või tekst "NULL". Siis ära tagasta midagi.
        if pd.isna(vastus) or vastus == '' or str(vastus).upper() == 'NULL':
            return None, ''
        # Kontrolli, kas vastus leidub sõnastikus
        if vastus in vastavused:
            # Kui jah, tagasta vastusele vastav kood
            return vastavused[vastus], ''
        else:
            # Kui ei, siis tagasta "muu" kood ja algne tekst
            return muu_kood, vastus
    
    # Rakenda mäpping
    result = data_coded[kysimus].apply(map_answer)
    data_coded[[kysimus, f'{kysimus}_tekst']] = result.apply(pd.Series)

In [316]:
def kodeeri_mitu_vastust(kysimus: str):
    """
    Kodeeri mitu vastust sisaldav lahter eraldi binaarsetesse veergudesse.
    Kasutab kõiki teadaolevaid vastuseid, mis võivad sisaldada koma.
    """
    mapping = vastuste_koodid[vastuste_koodid['Kysimus'] == kysimus][["Vastus", "Kood"]]
    vastavused = mapping.set_index('Vastus')['Kood'].to_dict()
    # Sort answers by length (longest first) to avoid partial matches
    sorted_answers = sorted(vastavused.keys(), key=len, reverse=True)

    # Find "muu" code
    muu_kood = None
    for v, k in vastavused.items():
        if v.strip().lower() == 'muu':
            muu_kood = k
            break

    # Initialize columns
    for vastus, kood in vastavused.items():
        veeru_nimi = f'{kysimus}_{kood}'
        if veeru_nimi not in data_coded.columns:
            data_coded[veeru_nimi] = 0
    if f'{kysimus}_muu_tekst' not in data_coded.columns:
        data_coded[f'{kysimus}_muu_tekst'] = ''

    for idx, lahtri_sisu in data_coded[kysimus].items():
        if pd.isna(lahtri_sisu) or lahtri_sisu == '' or str(lahtri_sisu).upper() == 'NULL':
            continue
        vastus_j22k = str(lahtri_sisu)
        found = []
        for ans in sorted_answers:
            if ans in vastus_j22k:
                kood = vastavused[ans]
                veeru_nimi = f'{kysimus}_{kood}'
                data_coded.loc[idx, veeru_nimi] = 1
                found.append(ans)
        # Remove found answers from the string
        for ans in found:
            vastus_j22k = vastus_j22k.replace(ans, '')
        # The rest (after removing commas and whitespace) is "muu"
        muu_tekst = ','.join([v.strip() for v in vastus_j22k.split(',') if v.strip()])
        if muu_tekst and muu_kood is not None:
            veeru_nimi = f'{kysimus}_{muu_kood}'
            data_coded.loc[idx, veeru_nimi] = 1
            data_coded.loc[idx, f'{kysimus}_muu_tekst'] = muu_tekst

# Example usage:
kodeeri_mitu_vastust('K23_kasutuskolbmatud_tekstiilid')


In [311]:
data_coded

,Vastaja_ID,K1_aeg,K2_kontroll,K3_vanus,K4_sugu,K5_elukoht,K6_keel,K7_sorteerimiskaitumine,K8_teadmiste_hinnang,K9_probleemi_tosidus,...,K19_loobumise_pohjused_8,K19_loobumise_pohjused_9,K19_loobumise_pohjused_muu_tekst,K23_kasutuskolbmatud_tekstiilid_1,K23_kasutuskolbmatud_tekstiilid_2,K23_kasutuskolbmatud_tekstiilid_3,K23_kasutuskolbmatud_tekstiilid_4,K23_kasutuskolbmatud_tekstiilid_5,K23_kasutuskolbmatud_tekstiilid_6,K23_kasutuskolbmatud_tekstiilid_muu_tekst
0,1,3/10/2025 8:01:13,Jah,18-29,Naine,Tallinn,Eesti keel,"Jah, maksimaalselt kahes kategoorias",5,5,...,0,0,,1,0,1,0,0,0,
1,2,3/10/2025 16:55:18,Jah,18-29,Naine,Pärnumaa,Eesti keel,"Jah, 3-5 jäätmekategoorias",3,3,...,0,0,,0,0,0,0,0,0,
2,3,3/17/2025 21:06:37,Jah,50-64,Naine,Harku vald,Eesti keel,"Jah, maksimaalselt kahes kategoorias",4,5,...,0,0,,1,0,1,0,0,0,
3,4,3/17/2025 20:00:21,Jah,50-64,Naine,Soome,Eesti keel,"Jah, 3-5 jäätmekategoorias",2,5,...,0,0,,0,1,0,0,0,0,
4,5,3/20/2025 21:15:13,Jah,18-29,Naine,Tartu maakond,Eesti keel,"Jah, 3-5 jäätmekategoorias",4,5,...,1,0,,1,1,0,0,0,0,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
673,674,3/28/2025 16:34:49,Jah,50-64,Naine,Pärnumaa,Eesti keel,"Jah, maksimaalselt kahes kategoorias",2,5,...,0,0,,1,0,0,0,0,0,
674,675,4/13/2025 9:46:43,Jah,18-29,Naine,harjumaa,Eesti keel,"Jah, üle 6 jäätmekategoorias",4,5,...,0,1,soov minimaliseerida rõivaste arvu,0,0,1,0,0,0,
675,676,4/15/2025 13:31:30,Jah,50-64,Naine,Harjumaa,Eesti keel,"Jah, 3-5 jäätmekategoorias",3,5,...,0,0,,0,1,0,0,0,0,
676,677,3/11/2025 19:48:03,Jah,30-49,Naine,Harjumaa,Eesti keel,"Jah, 3-5 jäätmekategoorias",3,4,...,0,0,,0,0,0,0,0,0,


In [318]:
data_coded.to_csv(
    "teisendus.csv",
    sep=';',
    columns=[
        'K23_kasutuskolbmatud_tekstiilid',
        'K23_kasutuskolbmatud_tekstiilid_1',
        'K23_kasutuskolbmatud_tekstiilid_2',
        'K23_kasutuskolbmatud_tekstiilid_3',
        'K23_kasutuskolbmatud_tekstiilid_4',
        'K23_kasutuskolbmatud_tekstiilid_5',
        'K23_kasutuskolbmatud_tekstiilid_6',
        'K23_kasutuskolbmatud_tekstiilid_7',
        'K23_kasutuskolbmatud_tekstiilid_muu_tekst'
    ],
    encoding='UTF-8')

## K2_kontroll

In [226]:
kodeeri_vastus('K2_kontroll')

## K3_vanus

In [ ]:
kodeeri_vastus('K3_vanus')

## K4_sugu

In [ ]:
kodeeri_vastus('K4_sugu')

## K5_elukoht

In [ ]:
def paranda_elukoha_vead(elukoht: str) -> str:
    '''
    Tagasta teadaolevate kirjavigadega elukoha nimetuse asemel korrektne nimetus, kust on elukoha eest ning järelt eemaldatud üleliigsed tühikud.
    '''
    elukoht = elukoht.lower().strip()
    
    # Paranda enamlevinud kirjavead
    asendused = {
        "tln": "tallinn",
        "tallinm": "tallinn",
        "harjumas": "harjumaa",
        "hafjumas": "harjumaa",
        "hatjumaa": "harjumaa",
        "pärjumaa": "pärnumaa",
        "jõheva": "jõgeva",
        "l-viru": "lääne-virumaa",
        "ida-virimaa": "ida-virumaa",
        "jarva maa": "järvamaa",
        "parnu": "pärnu",
        "järva jaani": "järva-jaani"
    }
    
    for k, v in asendused.items():
        elukoht = elukoht.replace(k, v)
    
    return elukoht

# Teadaolevate maakonna kirjapiltide vastavuspaarid
maakonnad = {
    "harjumaa": "Harju maakond",
    "harju maakond": "Harju maakond",
    "harju mk": "Harju maakond",
    "harju": "Harju maakond",
    "hiiumaa": "Hiiu maakond",
    "hiiu maakond": "Hiiu maakond",
    "ida-virumaa": "Ida-Viru maakond",
    "ida-viru maakond": "Ida-Viru maakond",
    "ida-viru": "Ida-Viru maakond",
    "jõgevamaa": "Jõgeva maakond",
    "jõgeva maakond": "Jõgeva maakond",
    "järvamaa": "Järva maakond",
    "järva maakond": "Järva maakond",
    "läänemaa": "Lääne maakond",
    "lääne maakond": "Lääne maakond",
    "lääne-virumaa": "Lääne-Viru maakond",
    "lääne-viru maakond": "Lääne-Viru maakond",
    "lääne-viru": "Lääne-Viru maakond",
    "põlvamaa": "Põlva maakond",
    "põlva maakond": "Põlva maakond",
    "pärnumaa": "Pärnu maakond",
    "pärnu maakond": "Pärnu maakond",
    "raplamaa": "Rapla maakond",
    "rapla maakond": "Rapla maakond",
    "saaremaa": "Saare maakond",
    "saare maakond": "Saare maakond",
    "tartumaa": "Tartu maakond",
    "tartu maakond": "Tartu maakond",
    "valgamaa": "Valga maakond",
    "valga maakond": "Valga maakond",
    "viljandimaa": "Viljandi maakond",
    "viljandi maakond": "Viljandi maakond",
    "võrumaa": "Võru maakond",
    "võru maakond": "Võru maakond"
}

# Linna ja maakonna vastavuspaarid
elukoht_maakond = {
    # Harjumaa
    "tallinn": "Harju maakond",
    "keila": "Harju maakond",
    "maardu": "Harju maakond",
    "paldiski": "Harju maakond",
    "viimsi": "Harju maakond",
    "saku": "Harju maakond",
    "saue": "Harju maakond",
    "rae": "Harju maakond",
    "harku": "Harju maakond",
    "lääne-harju": "Harju maakond",
    "klooga": "Harju maakond",
    "rummu": "Harju maakond",
    "padise": "Harju maakond",
    # Ida-Virumaa
    "jõhvi": "Ida-Viru maakond",
    # Jõgevamaa
    "jõgeva": "Jõgeva maakond",
    "luua": "Jõgeva maakond",
    "põltsamaa": "Jõgeva maakond",
    # Järvamaa
    "järva-jaani": "Järva maakond",
    "järva vald": "Järva maakond",
    # Lääne-Virumaa
    "rakvere": "Lääne-Viru maakond",
    # Põlvamaa
    "põlva": "Põlva maakond",
    # Pärnumaa
    "pärnu": "Pärnu maakond",
    "sindi": "Pärnu maakond",
    # Raplamaa
    "rapla": "Rapla maakond",
    "raka": "Rapla maakond",
    # Tartumaa
    "tartu": "Tartu maakond",
    "ülenurme": "Tartu maakond",
    "kambja": "Tartu maakond",
    # Valgamaa
    "valga": "Valga maakond",
    # Viljandimaa
    "viljandi": "Viljandi maakond",
    "põhja-sakala": "Viljandi maakond"
}

def tuleta_maakond(tekst: str) -> str:
    '''
    Tagasta etteantud sõnastike alusel vastaja elukoha maakond
    '''
    elukoht = paranda_elukoha_vead(tekst)
    elukohad = re.split(r"[,/]| ja ", elukoht)
    peamine_elukoht = elukohad[0].strip()

    # 1. direct county match
    for i in maakonnad:
        if i in peamine_elukoht:
            return maakonnad[i]

    # 2. place → county
    for i in elukoht_maakond:
        if i in peamine_elukoht:
            return elukoht_maakond[i]

    return "Muu"

data_coded['K5_elukoht'] = data_coded['K5_elukoht'].apply(tuleta_maakond)

kodeeri_vastus('K5_elukoht')

Vastus sõnastikust: Harju maakond
Vastus sõnastikust: Pärnu maakond
Vastus sõnastikust: Harju maakond
Vastus sõnastikust: Muu
Vastus sõnastikust: Tartu maakond
Vastus sõnastikust: Harju maakond
Vastus sõnastikust: Harju maakond
Vastus sõnastikust: Harju maakond
Vastus sõnastikust: Harju maakond
Vastus sõnastikust: Harju maakond
Vastus sõnastikust: Järva maakond
Vastus sõnastikust: Tartu maakond
Vastus sõnastikust: Ida-Viru maakond
Vastus sõnastikust: Harju maakond
Vastus sõnastikust: Harju maakond
Vastus sõnastikust: Harju maakond
Vastus sõnastikust: Harju maakond
Vastus sõnastikust: Tartu maakond
Vastus sõnastikust: Järva maakond
Vastus sõnastikust: Järva maakond
Vastus sõnastikust: Harju maakond
Vastus sõnastikust: Jõgeva maakond
Vastus sõnastikust: Tartu maakond
Vastus sõnastikust: Tartu maakond
Vastus sõnastikust: Tartu maakond
Vastus sõnastikust: Viljandi maakond
Vastus sõnastikust: Harju maakond
Vastus sõnastikust: Harju maakond
Vastus sõnastikust: Harju maakond
Vastus sõnastikus

## K6_keel

In [ ]:
kodeeri_vastus('K6_keel')

In [198]:
data_raw

,Vastaja_ID,K1_aeg,K2_kontroll,K3_vanus,K4_sugu,K5_elukoht,K6_keel,K7_sorteerimiskaitumine,K8_teadmiste_hinnang,K9_probleemi_tosidus,...,K34_ettepanekud,K35_valmisolek_telefonikoneks,K36_valmisolek_lisakysimusteks,K37_rahulolu_garderoobiga,K38_kandmise_kestus,K39_kandmise_kestus_tapsustus,K40_ostmissagedus,K41_ultrakiirmoe_ostmine,K42_ultrakiirmoe_eelistamise_pohjused,K43_roivaste_tellimine_proovimiseks
0,1,3/10/2025 8:01:13,Jah,18-29,Naine,Tallinn,Eesti keel,"Jah, maksimaalselt kahes kategoorias",5,5,...,NULL,ei,Jah,4,"Mu garderoobis on pikaealised riided, mida hoo...",NULL,Kord kvartalis,1,NULL,2
1,2,3/10/2025 16:55:18,Jah,18-29,Naine,Pärnumaa,Eesti keel,"Jah, 3-5 jäätmekategoorias",3,3,...,NULL,Ei,Jah,5,"Peaaegu ei loobugi, mul hetkel on ainus paar r...",NULL,Kord aastas,3,"Odavam hind, võid saada parema asja kui Eesti ...",1
2,3,3/17/2025 21:06:37,Jah,50-64,Naine,Harku vald,Eesti keel,"Jah, maksimaalselt kahes kategoorias",4,5,...,"Väga suur probleem on vanadel inimestel, nad e...",Ei,Ei,NULL,NULL,NULL,NULL,NULL,NULL,NULL
3,4,3/17/2025 20:00:21,Jah,50-64,Naine,Soome,Eesti keel,"Jah, 3-5 jäätmekategoorias",2,5,...,NULL,Ei,Jah,2,"Mu garderoobis on pikaealised riided, mida hoo...",NULL,Kord aastas,1,NULL,1
4,5,3/20/2025 21:15:13,Jah,18-29,Naine,Tartu maakond,Eesti keel,"Jah, 3-5 jäätmekategoorias",4,5,...,Sellest kirjutasin esimesel/teisel küsimuste l...,"Eelistaksin meili teel, eva.taigla@tartukunsti...",Jah,4,"Mu garderoobis on pikaealised riided, mida hoo...",NULL,Kord aastas,1,NULL,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
673,674,3/28/2025 16:34:49,Jah,50-64,Naine,Pärnumaa,Eesti keel,"Jah, maksimaalselt kahes kategoorias",2,5,...,NULL,-,Jah,3,"Mu garderoobis on pikaealised riided, mida hoo...",NULL,Kord kuus,1,NULL,5
674,675,4/13/2025 9:46:43,Jah,18-29,Naine,harjumaa,Eesti keel,"Jah, üle 6 jäätmekategoorias",4,5,...,NULL,ei,Jah,4,"Peamiselt kannan riideid aastaid, aga üksikute...",NULL,Kord kvartalis,2,NULL,3
675,676,4/15/2025 13:31:30,Jah,50-64,Naine,Harjumaa,Eesti keel,"Jah, 3-5 jäätmekategoorias",3,5,...,NULL,"Ei, aitäh",Jah,4,"Peamiselt kannan riideid aastaid, aga üksikute...",NULL,Kord kuus,3,NULL,1
676,677,3/11/2025 19:48:03,Jah,30-49,Naine,Harjumaa,Eesti keel,"Jah, 3-5 jäätmekategoorias",3,4,...,NULL,Ei,Jah,4,"Peamiselt kannan riideid aastaid, aga üksikute...",NULL,Kord kvartalis,2,"Kleid tegumood, suurus ja värv oli täpselt mid...",1


In [240]:
data_coded

,Vastaja_ID,K1_aeg,K2_kontroll,K3_vanus,K4_sugu,K5_elukoht,K6_keel,K7_sorteerimiskaitumine,K8_teadmiste_hinnang,K9_probleemi_tosidus,...,K35_valmisolek_telefonikoneks,K36_valmisolek_lisakysimusteks,K37_rahulolu_garderoobiga,K38_kandmise_kestus,K39_kandmise_kestus_tapsustus,K40_ostmissagedus,K41_ultrakiirmoe_ostmine,K42_ultrakiirmoe_eelistamise_pohjused,K43_roivaste_tellimine_proovimiseks,K5_elukoht_tekst
0,1,3/10/2025 8:01:13,Jah,18-29,Naine,16,Eesti keel,"Jah, maksimaalselt kahes kategoorias",5,5,...,ei,Jah,4,"Mu garderoobis on pikaealised riided, mida hoo...",NULL,Kord kvartalis,1,NULL,2,Tallinn
1,2,3/10/2025 16:55:18,Jah,18-29,Naine,16,Eesti keel,"Jah, 3-5 jäätmekategoorias",3,3,...,Ei,Jah,5,"Peaaegu ei loobugi, mul hetkel on ainus paar r...",NULL,Kord aastas,3,"Odavam hind, võid saada parema asja kui Eesti ...",1,Pärnumaa
2,3,3/17/2025 21:06:37,Jah,50-64,Naine,16,Eesti keel,"Jah, maksimaalselt kahes kategoorias",4,5,...,Ei,Ei,NULL,NULL,NULL,NULL,NULL,NULL,NULL,Harku vald
3,4,3/17/2025 20:00:21,Jah,50-64,Naine,16,Eesti keel,"Jah, 3-5 jäätmekategoorias",2,5,...,Ei,Jah,2,"Mu garderoobis on pikaealised riided, mida hoo...",NULL,Kord aastas,1,NULL,1,Soome
4,5,3/20/2025 21:15:13,Jah,18-29,Naine,12,Eesti keel,"Jah, 3-5 jäätmekategoorias",4,5,...,"Eelistaksin meili teel, eva.taigla@tartukunsti...",Jah,4,"Mu garderoobis on pikaealised riided, mida hoo...",NULL,Kord aastas,1,NULL,1,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
673,674,3/28/2025 16:34:49,Jah,50-64,Naine,16,Eesti keel,"Jah, maksimaalselt kahes kategoorias",2,5,...,-,Jah,3,"Mu garderoobis on pikaealised riided, mida hoo...",NULL,Kord kuus,1,NULL,5,Pärnumaa
674,675,4/13/2025 9:46:43,Jah,18-29,Naine,16,Eesti keel,"Jah, üle 6 jäätmekategoorias",4,5,...,ei,Jah,4,"Peamiselt kannan riideid aastaid, aga üksikute...",NULL,Kord kvartalis,2,NULL,3,harjumaa
675,676,4/15/2025 13:31:30,Jah,50-64,Naine,16,Eesti keel,"Jah, 3-5 jäätmekategoorias",3,5,...,"Ei, aitäh",Jah,4,"Peamiselt kannan riideid aastaid, aga üksikute...",NULL,Kord kuus,3,NULL,1,Harjumaa
676,677,3/11/2025 19:48:03,Jah,30-49,Naine,16,Eesti keel,"Jah, 3-5 jäätmekategoorias",3,4,...,Ei,Jah,4,"Peamiselt kannan riideid aastaid, aga üksikute...",NULL,Kord kvartalis,2,"Kleid tegumood, suurus ja värv oli täpselt mid...",1,Harjumaa


# Kontroll!

In [264]:
# Kontroll
duckdb.sql('''
    SELECT DISTINCT
        K5_elukoht
        --,K5_elukoht_tekst
        ,count(*) as vastuste_arv
    FROM data_coded
    group by
        K5_elukoht
        --,K5_elukoht_tekst
    ORDER BY K5_elukoht
''').df()

,K5_elukoht,vastuste_arv
0,1,408
1,2,2
2,3,3
3,4,36
4,5,7
5,6,1
6,7,12
7,8,6
8,9,57
9,10,13


In [220]:
duckdb.sql('''
    SELECT DISTINCT
        t1.K4_sugu,
        t2.Kood,
        count(t1.*) as vastuste_arv
    FROM data_raw t1
    LEFT JOIN vastuste_koodid t2 on t1.K4_sugu = t2.Vastus
        AND t2.kysimus = 'K4_sugu'
    GROUP BY t1.K4_sugu, t2.Kood
    ORDER BY t2.Kood
''').df()

,K4_sugu,Kood,vastuste_arv
0,Mees,1,73
1,Naine,2,596
2,Ei soovi määratleda,3,9
